In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import joblib

In [2]:
df = pd.read_csv(r"C:\Users\ADEEBSAYEED\Downloads\fitness_class_2212.csv")
print("Data loaded successfully!")
print(df.head())


Data loaded successfully!
   booking_id  months_as_member  weight days_before day_of_week time  \
0           1                17   79.56           8         Wed   PM   
1           2                10   79.01           2         Mon   AM   
2           3                16   74.53          14         Sun   AM   
3           4                 5   86.12          10         Fri   AM   
4           5                15   69.29           8         Thu   AM   

   category  attended  
0  Strength         0  
1      HIIT         0  
2  Strength         0  
3   Cycling         0  
4      HIIT         0  


In [3]:
df.drop_duplicates(inplace=True)
df.dropna(subset=['attended'], inplace=True)
df['attended'] = df['attended'].astype(int)

print(df['attended'].value_counts())

attended
0    1046
1     454
Name: count, dtype: int64


In [4]:
X = df.drop(columns=['attended'])
y = df['attended']

In [5]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(exclude=['object']).columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)

Categorical: ['days_before', 'day_of_week', 'time', 'category']
Numerical: ['booking_id', 'months_as_member', 'weight']


In [6]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [8]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, class_weight="balanced"))
])

In [9]:
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [5, 10, 15, None],
    "classifier__min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(
    model, param_grid, cv=3, scoring="f1", n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("✅ Best model found:", grid_search.best_params_)


Fitting 3 folds for each of 24 candidates, totalling 72 fits
✅ Best model found: {'classifier__max_depth': 5, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}


In [10]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("\n🔹 Accuracy:", accuracy_score(y_test, y_pred))
print("🔹 ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


🔹 Accuracy: 0.7666666666666667
🔹 ROC-AUC: 0.794048057205952

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.78      0.82       209
           1       0.59      0.75      0.66        91

    accuracy                           0.77       300
   macro avg       0.73      0.76      0.74       300
weighted avg       0.79      0.77      0.77       300


Confusion Matrix:
 [[162  47]
 [ 23  68]]


In [11]:
joblib.dump(best_model, "best_fitness_rf_model.joblib")
print(" Model saved as best_fitness_rf_model.joblib")

 Model saved as best_fitness_rf_model.joblib
